# Exploration

Scratch notebook for the HEST-Benchmark IDC task. Nothing here is load-bearing
— anything that matters should move into `src/`.

In [ ]:
import sys

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from data_loading import BENCH_ROOT, N_FOLDS, load_gene_list, load_split, read_expression, read_patch_barcodes
from embed_images import load_cached_embeddings

ROOT = "../" + str(BENCH_ROOT)
genes = load_gene_list(ROOT)
len(genes), genes[:8]

In [ ]:
for fold in range(N_FOLDS):
    tr = load_split(fold, "train", ROOT).sample_id.tolist()
    te = load_split(fold, "test", ROOT).sample_id.tolist()
    print(f"fold {fold}: train {tr} test {te}")

In [ ]:
# Expression for one sample. log1p only, matching the benchmark.
sid = "TENX95"
barcodes = read_patch_barcodes(f"{ROOT}/patches/{sid}.h5")
y = read_expression(f"{ROOT}/adata/{sid}.h5ad", genes, barcodes)
X = load_cached_embeddings([f"{sid}:{b}" for b in barcodes], root="../embeddings")
X.shape, y.shape

In [ ]:
# Which genes vary most? Flat ones cap the achievable Pearson.
order = np.argsort(y.std(axis=0))[::-1]
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(genes)), y.std(axis=0)[order])
ax.set_xticks(range(len(genes)))
ax.set_xticklabels([genes[i] for i in order], rotation=90, fontsize=6)
ax.set_ylabel("std of log1p counts")
plt.tight_layout()

In [ ]:
# Per-gene Pearson from the last benchmark run, worst genes first.
import json

res = json.load(open("../results/benchmark_idc.json"))
per_gene = res["folds"][0]["per_gene"]
sorted(per_gene.items(), key=lambda kv: kv[1])[:10]